# 04 - Sampler 采样器

## 学习目标

1. 理解 Sampler 的作用
2. 掌握常用的 Sampler 类型
3. 学会使用 WeightedRandomSampler 处理不平衡数据
4. 了解如何自定义 Sampler

## 1. 什么是 Sampler？

Sampler（采样器）控制 DataLoader 如何从数据集中采样数据。

**DataLoader 的数据获取流程：**

```
Sampler 生成索引 → DataLoader 根据索引获取数据 → 返回 batch
```

**关键理解：**

- `shuffle=True` 实际上是使用了 `RandomSampler`
- `shuffle=False` 实际上是使用了 `SequentialSampler`
- `shuffle` 和 `sampler` 参数互斥

## 2. PyTorch 内置的 Sampler

| Sampler | 说明 | 使用场景 |
|---------|------|----------|
| `SequentialSampler` | 顺序采样 | 验证集、测试集 |
| `RandomSampler` | 随机采样 | 训练集 |
| `WeightedRandomSampler` | 加权随机采样 | 类别不平衡 |
| `SubsetRandomSampler` | 子集随机采样 | 交叉验证 |
| `BatchSampler` | 批量采样 | 自定义批次 |

## 3. SequentialSampler：顺序采样

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, SequentialSampler

class SimpleDataset(Dataset):
    def __init__(self, n):
        self.data = list(range(n))
    
    def __getitem__(self, idx):
        return self.data[idx]
    
    def __len__(self):
        return len(self.data)

dataset = SimpleDataset(10)

# 使用 SequentialSampler
sampler = SequentialSampler(dataset)
loader = DataLoader(dataset, batch_size=3, sampler=sampler)

print("SequentialSampler（顺序采样）：\n")
for batch_idx, batch in enumerate(loader):
    print(f"Batch {batch_idx}: {batch}")

print("\n等价于 shuffle=False")

## 4. RandomSampler：随机采样

In [ ]:
from torch.utils.data import RandomSampler

dataset = SimpleDataset(10)

# 使用 RandomSampler
sampler = RandomSampler(dataset)
loader = DataLoader(dataset, batch_size=3, sampler=sampler)

print("RandomSampler（随机采样）：\n")
for batch_idx, batch in enumerate(loader):
    print(f"Batch {batch_idx}: {batch}")

print("\n每次遍历顺序都不同：")
for batch_idx, batch in enumerate(loader):
    print(f"Batch {batch_idx}: {batch}")

print("\n等价于 shuffle=True")

## 5. WeightedRandomSampler：加权随机采样

### 5.1 为什么需要加权采样？

在实际应用中，数据集经常存在**类别不平衡**问题：

```
例如：医疗诊断数据集
- 健康样本：900 个
- 患病样本：100 个

问题：
模型会倾向于预测"健康"，因为这样准确率就有 90%！
但我们更关心能否正确识别"患病"样本。
```

**解决方案：**

使用 `WeightedRandomSampler`，给少数类别更高的采样权重。

### 5.2 WeightedRandomSampler 基础示例

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import collections

class ImbalancedDataset(Dataset):
    """不平衡数据集"""
    
    def __init__(self):
        # 类别 0：8 个样本
        # 类别 1：2 个样本
        self.data = [0, 0, 0, 0, 0, 0, 0, 0, 1, 1]
    
    def __getitem__(self, idx):
        return self.data[idx]
    
    def __len__(self):
        return len(self.data)

dataset = ImbalancedDataset()

print("原始数据分布：")
print(collections.Counter(dataset.data))
print("类别 0: 8 个, 类别 1: 2 个")
print("比例：4:1（严重不平衡）\n")

### 5.3 不使用 Sampler 的效果

In [ ]:
# 不使用 sampler
loader_normal = DataLoader(dataset, batch_size=2, shuffle=True)

print("不使用 Sampler（随机打乱）：")
sampled = []
for batch in loader_normal:
    sampled.extend(batch.tolist())

print(f"采样结果: {sampled}")
print(f"类别分布: {collections.Counter(sampled)}")
print("类别 0 仍然占多数\n")

### 5.4 使用 WeightedRandomSampler

In [ ]:
# 第一步：定义每个类别的权重
# 类别 0：权重 1
# 类别 1：权重 4（希望类别 1 被采样的概率是类别 0 的 4 倍）
class_weights = torch.tensor([1.0, 4.0])

# 第二步：为每个样本分配权重
sample_weights = [class_weights[label] for label in dataset.data]
print("每个样本的权重：")
print(sample_weights)
print()

# 第三步：创建 WeightedRandomSampler
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True  # 有放回采样
)

# 第四步：创建 DataLoader
loader_weighted = DataLoader(
    dataset,
    batch_size=2,
    sampler=sampler  # 注意：不能同时使用 shuffle
)

print("使用 WeightedRandomSampler：")
sampled = []
for batch in loader_weighted:
    sampled.extend(batch.tolist())

print(f"采样结果: {sampled}")
print(f"类别分布: {collections.Counter(sampled)}")
print("类别 1 的采样频率大大增加！")

### 5.5 实际案例：处理严重不平衡的数据集

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import collections

class MedicalDataset(Dataset):
    """模拟医疗数据集（严重不平衡）"""
    
    def __init__(self):
        # 类别 0（健康）：90 个
        # 类别 1（患病）：10 个
        self.labels = [0] * 90 + [1] * 10
    
    def __getitem__(self, idx):
        # 返回 (数据, 标签)
        return torch.randn(10), self.labels[idx]
    
    def __len__(self):
        return len(self.labels)

dataset = MedicalDataset()

print("原始数据集：")
print(f"总样本数: {len(dataset)}")
label_counter = collections.Counter(dataset.labels)
print(f"类别分布: {label_counter}")
print(f"不平衡比例: {label_counter[0]}:{label_counter[1]}\n")

# 计算类别权重（使用倒数）
class_counts = [label_counter[i] for i in sorted(label_counter)]
class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
print(f"类别权重: {class_weights}")
print(f"类别 1 的权重是类别 0 的 {class_weights[1]/class_weights[0]:.1f} 倍\n")

# 为每个样本分配权重
sample_weights = [class_weights[label] for label in dataset.labels]

# 创建 WeightedRandomSampler
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# 创建 DataLoader
loader = DataLoader(dataset, batch_size=10, sampler=sampler)

# 统计采样结果
print("采样 500 个样本的类别分布：")
sampled_labels = []
for i, (data, labels) in enumerate(loader):
    sampled_labels.extend(labels.tolist())
    if i >= 49:  # 50 个 batch = 500 个样本
        break

sampled_counter = collections.Counter(sampled_labels)
print(f"采样分布: {sampled_counter}")
print(f"采样比例: {sampled_counter[0]}:{sampled_counter[1]}")
print("\n通过加权采样，类别分布更加平衡！")

### 5.6 自动计算权重的通用方法

In [ ]:
def get_weighted_sampler(dataset_labels):
    """
    为不平衡数据集创建 WeightedRandomSampler
    
    参数:
        dataset_labels: 数据集的所有标签列表
    
    返回:
        WeightedRandomSampler
    """
    # 1. 统计每个类别的样本数
    label_counter = collections.Counter(dataset_labels)
    
    # 2. 计算类别权重（使用倒数）
    num_classes = len(label_counter)
    class_weights = torch.zeros(num_classes)
    for label, count in label_counter.items():
        class_weights[label] = 1.0 / count
    
    # 3. 为每个样本分配权重
    sample_weights = [class_weights[label] for label in dataset_labels]
    
    # 4. 创建 sampler
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )
    
    return sampler

# 使用示例
labels = [0] * 80 + [1] * 15 + [2] * 5  # 3 个类别，严重不平衡
print(f"原始分布: {collections.Counter(labels)}")

sampler = get_weighted_sampler(labels)
print("\n创建了 WeightedRandomSampler")
print("可以直接用于 DataLoader！")

## 6. SubsetRandomSampler：子集随机采样

In [ ]:
from torch.utils.data import SubsetRandomSampler

# 创建数据集
dataset = SimpleDataset(20)

# 定义子集索引
train_indices = list(range(0, 15))    # 前 15 个
valid_indices = list(range(15, 20))   # 后 5 个

# 创建两个 sampler
train_sampler = SubsetRandomSampler(train_indices)
valid_sampler = SubsetRandomSampler(valid_indices)

# 创建两个 DataLoader
train_loader = DataLoader(dataset, batch_size=5, sampler=train_sampler)
valid_loader = DataLoader(dataset, batch_size=5, sampler=valid_sampler)

print("训练集（索引 0-14）：")
for batch in train_loader:
    print(f"  {batch}")

print("\n验证集（索引 15-19）：")
for batch in valid_loader:
    print(f"  {batch}")

print("\n用途：交叉验证、划分训练集/验证集")

## 7. 自定义 Sampler

In [ ]:
from torch.utils.data import Sampler

class ReverseSampler(Sampler):
    """反向采样：从后往前"""
    
    def __init__(self, data_source):
        self.data_source = data_source
    
    def __iter__(self):
        # 返回索引的迭代器
        return iter(range(len(self.data_source) - 1, -1, -1))
    
    def __len__(self):
        return len(self.data_source)

# 测试
dataset = SimpleDataset(10)
sampler = ReverseSampler(dataset)
loader = DataLoader(dataset, batch_size=3, sampler=sampler)

print("ReverseSampler（反向采样）：")
for batch in loader:
    print(f"  {batch}")

### 自定义 Sampler：每个类别采样固定数量

In [ ]:
import random

class BalancedBatchSampler(Sampler):
    """每个 batch 包含每个类别的固定数量样本"""
    
    def __init__(self, labels, n_classes, n_samples_per_class):
        """
        参数:
            labels: 所有样本的标签
            n_classes: 类别数量
            n_samples_per_class: 每个类别每个 batch 采样多少个
        """
        self.labels = labels
        self.n_classes = n_classes
        self.n_samples_per_class = n_samples_per_class
        
        # 为每个类别建立索引
        self.label_to_indices = {}
        for label in range(n_classes):
            self.label_to_indices[label] = [
                i for i, l in enumerate(labels) if l == label
            ]
    
    def __iter__(self):
        # 每个类别随机采样
        indices = []
        for label in range(self.n_classes):
            label_indices = self.label_to_indices[label]
            sampled = random.sample(
                label_indices,
                min(self.n_samples_per_class, len(label_indices))
            )
            indices.extend(sampled)
        
        # 打乱
        random.shuffle(indices)
        return iter(indices)
    
    def __len__(self):
        return self.n_classes * self.n_samples_per_class

# 测试
labels = [0] * 20 + [1] * 10  # 不平衡数据集
sampler = BalancedBatchSampler(
    labels=labels,
    n_classes=2,
    n_samples_per_class=5
)

print("BalancedBatchSampler：")
print(f"每次采样 {len(sampler)} 个样本")
print(f"每个类别 {5} 个\n")

for epoch in range(2):
    print(f"Epoch {epoch + 1}:")
    sampled_indices = list(iter(sampler))
    sampled_labels = [labels[i] for i in sampled_indices]
    print(f"  采样的标签: {collections.Counter(sampled_labels)}")

## 8. Sampler 使用总结

### 8.1 何时使用哪种 Sampler？

| 场景 | 推荐 Sampler | 说明 |
|------|-------------|------|
| 训练集（平衡数据） | shuffle=True | 等价于 RandomSampler |
| 验证集/测试集 | shuffle=False | 等价于 SequentialSampler |
| 类别不平衡 | WeightedRandomSampler | 给少数类别更高权重 |
| 交叉验证 | SubsetRandomSampler | 划分训练集/验证集 |
| 特殊需求 | 自定义 Sampler | 灵活控制采样策略 |

### 8.2 常见错误

In [ ]:
# ❌ 错误 1：shuffle 和 sampler 同时使用
# loader = DataLoader(
#     dataset,
#     batch_size=32,
#     shuffle=True,        # 错误！
#     sampler=my_sampler   # 错误！
# )

# ✅ 正确：二选一
loader1 = DataLoader(dataset, batch_size=32, shuffle=True)
loader2 = DataLoader(dataset, batch_size=32, sampler=my_sampler)

print("shuffle 和 sampler 互斥，只能选一个！")

In [ ]:
# ❌ 错误 2：忘记设置 replacement=True
# 如果样本权重差异很大，不使用放回采样可能导致某些样本永远采样不到

# ✅ 正确：使用 replacement=True
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True  # 重要！
)

print("WeightedRandomSampler 建议使用 replacement=True")

## 9. 练习题

### 练习 1：处理不平衡数据集

创建一个不平衡数据集（类别 0: 100 个，类别 1: 20 个，类别 2: 5 个），使用 WeightedRandomSampler 进行平衡采样

In [ ]:
# TODO: 创建不平衡数据集
# labels = [0] * 100 + [1] * 20 + [2] * 5

# TODO: 计算类别权重

# TODO: 创建 WeightedRandomSampler

# TODO: 创建 DataLoader 并验证采样分布

### 练习 2：实现交叉验证

使用 SubsetRandomSampler 实现 5-fold 交叉验证

In [ ]:
# TODO: 创建数据集（100 个样本）

# TODO: 划分 5 个 fold

# TODO: 为每个 fold 创建训练集和验证集的 Sampler

## 10. 小结

### 核心要点

1. **Sampler 的作用**：控制 DataLoader 的采样策略
2. **常用 Sampler**：
   - `SequentialSampler`：顺序采样
   - `RandomSampler`：随机采样
   - `WeightedRandomSampler`：加权随机采样（处理不平衡数据）
   - `SubsetRandomSampler`：子集采样（交叉验证）
3. **shuffle 与 sampler**：互斥，不能同时使用

### WeightedRandomSampler 使用步骤

```python
# 1. 计算类别权重（使用倒数）
class_weights = 1.0 / torch.tensor(class_counts)

# 2. 为每个样本分配权重
sample_weights = [class_weights[label] for label in labels]

# 3. 创建 sampler
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# 4. 创建 DataLoader
loader = DataLoader(dataset, batch_size=32, sampler=sampler)
```

### 下一步

在下一个 notebook 中，我们将学习 **数据变换（Transforms）**，掌握数据预处理和数据增强技术！

---

## 📝 练习题

完成本章学习后，请尝试 [exercises.md](./exercises.md) 中的练习题来巩固知识。

**建议：**
- 先完成基础题，确保理解核心概念
- 进阶题帮助你深入掌握技巧
- 挑战题锻炼综合应用能力

💪 加油！实践是最好的学习方式！